# Cognito — Sistema de Inferência com Paging Dinâmico de KV Cache em Nível de Aplicação

**Trabalho de Conclusão de Curso — Anexo Técnico de Implementação**

Este notebook documenta a implementação de referência do sistema Cognito. O pipeline é organizado em três fases executadas em ambientes isolados:

| Fase | Script | Descrição |
|------|--------|-----------|
| 1 | `1_ingestao.py` | Extração do corpus TriviaQA e indexação vetorial via ChromaDB |
| 2 | `2_avaliacao.py` | Benchmarking padronizado (ARC, HellaSwag, WinoGrande, MMLU, TriviaQA) via LM Evaluation Harness |
| 3 | `3_inferencia.py` | Pipeline RAG completo com `VirtualPageManager` e avaliação comparativa de pico de VRAM |

**Ambiente de referência:** Google Colab — GPU NVIDIA T4 (16 GB VRAM)  
**Modelo:** `mistralai/Mistral-7B-Instruct-v0.3` — quantização NF4 (bitsandbytes)  
**Bibliotecas:** HuggingFace Transformers ≥ 4.46, bitsandbytes, ChromaDB, sentence-transformers, lm-eval

---

**Justificativa Arquitetural — `%%writefile` e Isolamento de Kernel:**  
Cada fase é escrita via `%%writefile` e executada como script autônomo pelo gerenciador `uv` (`!uv run script.py`). Essa decisão garante dois requisitos científicos críticos:

1. **Isolamento de Estado (VRAM):** O interpretador IPython retém tensores na GPU entre células. A execução como processo separado garante que a VRAM seja integralmente liberada ao término de cada fase.
2. **Reproducibilidade de Dependências:** Cada fase declara suas dependências via PEP 723 (inline script metadata), permitindo que o `uv` instale um ambiente exato e isolado — eliminando conflitos entre bibliotecas como `bitsandbytes`, `lm-eval` e `torchvision` presentes na imagem base do Colab.


## 0. Gestão de Sessão — Google Drive

Execute a **Célula de Início** ao abrir o Colab e a **Célula de Fim** antes de fechar.

| O que persiste no Drive | Por quê |
|-------------------------|----------|
| `chroma_cognito/` | ChromaDB — caro de reconstruir (~10 min embedding) |
| `gold_answers.json` | Queries de avaliação e gabaritos |
| `results_checkpoint_C*.jsonl` | Resultados — perder esses é perder sessões de GPU |
| Modelo Mistral | **Não persiste** — HF Hub re-download (~1 min) é mais rápido que Drive |

> **Estratégia de I/O:** ChromaDB usa SQLite com leituras aleatórias intensas.  
> Rodar direto do Drive seria 3-5× mais lento. A cópia Drive → `/content/` leva ~15-30s e elimina o gargalo.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE INÍCIO DE SESSÃO — Execute SEMPRE ao abrir o Colab
# ═══════════════════════════════════════════════════════════════
# Monta o Drive e copia dados persistentes para o SSD local (/content/).
# ChromaDB tem I/O aleatório intenso — rodar direto do Drive seria
# 3-5x mais lento. A cópia local demora ~10-30s e resolve o problema.

import os, shutil, time
from google.colab import drive

drive.mount('/content/drive')

# ── Configuração de caminhos ─────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT  = '/content'

# Cria diretórios no Drive se não existirem
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/chroma_cognito', exist_ok=True)

# ── Restaura ChromaDB (índice vetorial) ──────────────────────────
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
chroma_sqlite = f'{DRIVE_CHROMA}/chroma.sqlite3'

if os.path.exists(chroma_sqlite):
    print('Restaurando ChromaDB do Drive → /content/ ...')
    t0 = time.time()
    if os.path.exists(LOCAL_CHROMA):
        shutil.rmtree(LOCAL_CHROMA)
    shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(LOCAL_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB restaurado: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
else:
    print('⚠ ChromaDB não encontrado no Drive.')
    print('  Execute a célula de ingestão (1_ingestao.py) para construí-lo.')

# ── Restaura gold_answers.json ────────────────────────────────────
DRIVE_GOLD = f'{DRIVE_ROOT}/gold_answers.json'
if os.path.exists(DRIVE_GOLD):
    shutil.copy(DRIVE_GOLD, f'{LOCAL_ROOT}/gold_answers.json')
    import json as _j
    n = len(_j.load(open(f'{LOCAL_ROOT}/gold_answers.json')))
    print(f'  ✓ gold_answers.json restaurado: {n} queries')
else:
    print('  ⚠ gold_answers.json não encontrado — será gerado pela ingestão.')

# ── Restaura checkpoints de runs anteriores ───────────────────────
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'
ckpts = [f for f in os.listdir(DRIVE_CKPTS) if f.endswith('.jsonl')]
for ck in ckpts:
    shutil.copy(f'{DRIVE_CKPTS}/{ck}', f'{LOCAL_ROOT}/{ck}')
if ckpts:
    print(f'  ✓ Checkpoints restaurados: {ckpts}')
else:
    print('  ℹ Nenhum checkpoint anterior encontrado (primeira sessão).')

print()
print('✓ Sessão restaurada. Dados em /content/ prontos para uso.')
print(f'  Drive root: {DRIVE_ROOT}')


# TODO:
- [x] **Está consumado...**

## 1. Instalação de Dependências

In [ ]:
%%writefile 1_ingestao.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "datasets>=2.20.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "torch",
#     "einops"
# ]
# ///

import json
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
VECTOR_DB_PATH       = "./chroma_cognito"

# ── Separação Corpus / Avaliação (Fase 0 — Roadmap) ──────────────────────────
# Documentos indexados vêm do split TRAIN do TriviaQA.
# As queries de avaliação vêm do split VALIDATION (conjunto independente).
# Isso elimina data leakage: o modelo não pode "lembrar" contextos que estão
# no mesmo split das perguntas de teste.
CORPUS_SPLIT     = "train[:600]"      # Documentos para o índice vetorial
EVAL_SPLIT       = "validation[:200]" # Perguntas de avaliação (independentes)

MIN_PASSAGE_CHARS   = 500   # Mantém apenas passagens substantivas
MAX_CORPUS_PASSAGES = 1000  # Limite superior do corpus indexado


def main():
    # ── Corpus (train) ───────────────────────────────────────────────────────
    print("Carregando corpus TriviaQA (split TRAIN para indexação)...")
    train_data = load_dataset("trivia_qa", "rc.wikipedia", split=CORPUS_SPLIT)

    # ── [M1] Chunking de passagens para retrieval preciso ──────────────
    # Passagens Wikipedia inteiras (~5.000-30.000 chars) geram retrieval
    # impreciso. Chunks de 1024 chars (~256 tokens) com overlap de 256
    # chars permitem retrieval cirúrgico por parágrafo.
    CHUNK_CHARS   = 1024
    OVERLAP_CHARS = 256

    def chunk_passage(text, chunk_chars=CHUNK_CHARS, overlap=OVERLAP_CHARS):
        chunks, start = [], 0
        while start < len(text):
            end   = min(start + chunk_chars, len(text))
            chunk = text[start:end].strip()
            if len(chunk) >= 150:
                chunks.append(chunk)
            if end == len(text):
                break
            start = end - overlap
        return chunks if chunks else [text.strip()]

    raw_passages = []
    for example in train_data:
        for passage in example["entity_pages"]["wiki_context"]:
            if passage and len(passage.strip()) >= MIN_PASSAGE_CHARS:
                raw_passages.append(passage.strip())

    # Deduplicação por hash de prefixo (512 chars)
    seen, dedup = set(), []
    for doc in raw_passages:
        key = doc[:512]
        if key not in seen:
            seen.add(key)
            dedup.append(doc)

    # Chunking com overlap
    documents = []
    for doc in dedup:
        documents.extend(chunk_passage(doc))
    documents = documents[:MAX_CORPUS_PASSAGES]

    print(f"Corpus: {len(documents)} chunks indexados (de {len(dedup)} passagens únicas, split=TRAIN).")
    print(f"Comprimento médio: {sum(len(d) for d in documents) // max(len(documents),1)} chars.")

    # ── Gold Answers (validation) ─────────────────────────────────────────────
    print("Carregando queries de avaliação (split VALIDATION — independente do corpus)...")
    val_data = load_dataset("trivia_qa", "rc.wikipedia", split=EVAL_SPLIT)

    gold_answers = {}
    for example in val_data:
        question = example["question"]
        gold_answers[question] = example["answer"]["aliases"]

    print(f"Queries de avaliação disponíveis: {len(gold_answers)} (split=VALIDATION).")

    with open("gold_answers.json", "w", encoding="utf-8") as f:
        json.dump(gold_answers, f, ensure_ascii=False)

    # ── Indexação ─────────────────────────────────────────────────────────────
    print("Calculando embeddings e escrevendo no DB local...")
    client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
    try:
        client.delete_collection("cognito_knowledge_base")
    except Exception:
        pass

    collection = client.create_collection(
        name="cognito_knowledge_base",
        metadata={"hnsw:space": "cosine"}
    )

    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
    embeddings  = embed_model.encode(
        documents,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    collection.add(
        documents=documents,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(documents))],
    )
    print("Fase 1 concluída com sucesso.")
    print(f"  Corpus (TRAIN):      {len(documents)} passagens indexadas.")
    print(f"  Avaliação (VAL):     {len(gold_answers)} queries com gabarito.")
    print("  Data leakage: ZERO — splits são disjuntos por design.")


if __name__ == "__main__":
    main()





In [ ]:
!pip install -q uv
!uv run 1_ingestao.py


## Fase 2: Avaliação Padronizada (LM-Eval)
Roda o pipeline oficial no ambiente limpo e é desmantelado depois.


In [ ]:

%%writefile 2_avaliacao.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "lm-eval>=0.4.4",
#     "transformers>=4.46.0",
#     "accelerate>=0.34.0",
#     "bitsandbytes>=0.44.1",
#     "torch",
#     "einops"
# ]
# ///

import torch
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Carregando modelo NF4 para avaliação padronizada...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Passar o modelo já instanciado para o HFLM evita conflito de args internos
lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=4)

print('Executando benchmarks padronizados...')
results = lm_eval.simple_evaluate(
    model=lm,
    tasks=['arc_challenge', 'hellaswag', 'winogrande', 'mmlu', 'triviaqa'],
    num_fewshot=0,
    limit=500,
    log_samples=True,
)

print(lm_eval.utils.make_table(results))


In [ ]:
# Execução completa requer ~40-60 min em GPU dedicada (A100/V100).
# Em Colab T4 gratuito o tempo excede o limite de sessão.
# Os resultados abaixo são reportados por referência cruzada —
# Jiang et al. (2023) e HuggingFace Open LLM Leaderboard.
# Quantização NF4 introduz degradação < 2% (Dettmers et al., 2023).
#
# Para reprodução completa, descomente a linha abaixo:
# !uv run 2_avaliacao.py

REFERENCE_RESULTS = {
    "arc_challenge": {"metric": "acc_norm", "score": 0.5998, "source": "HF Open LLM Leaderboard"},
    "hellaswag":     {"metric": "acc_norm", "score": 0.8130, "source": "HF Open LLM Leaderboard"},
    "mmlu":          {"metric": "acc",      "score": 0.6010, "source": "Jiang et al. (2023)"},
    "winogrande":    {"metric": "acc",      "score": 0.7530, "source": "HF Open LLM Leaderboard"},
}

print("Fase 2 — Mistral-7B-Instruct-v0.3 (float16, referência cruzada)")
print(f"\n{'Benchmark':<20} {'Métrica':<12} {'Score':>8}  Fonte")
print("─" * 72)
for task, d in REFERENCE_RESULTS.items():
    print(f"{task:<20} {d['metric']:<12} {d['score']:>8.4f}  {d['source']}")

## Fase 3: RAG Cognito & Avaliação de Paging
Uma máquina com 100% de VRAM dedicada apenas à Inferência Paging Core.


In [ ]:
%%writefile 3_inferencia.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "numpy>=1.26.4",
#     "transformers>=4.46.0",
#     "accelerate>=0.34.0",
#     "bitsandbytes>=0.44.1",
#     "peft>=0.12.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "scikit-learn>=1.5.1",
#     "torch",
#     "einops"
# ]
# ///

import gc
import json
import math
import os
import sys
import time
import torch
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# Constantes
# -----------------------------------------------------------------------------
EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
RERANKER_MODEL_NAME  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL_NAME       = "mistralai/Mistral-7B-Instruct-v0.3"
VECTOR_DB_PATH       = "./chroma_cognito"

MAX_CONTEXT_CHARS   = 40_000
MAX_NEW_TOKENS      = 128
NUM_QUERIES_EVAL    = 50

PAGING_CONTEXT_THRESHOLD_TOKENS = 1000
CHUNKED_PREFILL_CHUNK_SIZE      = 512
CHUNKED_PREFILL_THRESHOLD_TOKENS = PAGING_CONTEXT_THRESHOLD_TOKENS

def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def _get_kv_tensors(cache):
    """Extract key/value tensor lists from any DynamicCache format.
    Handles: (a) legacy .key_cache/.value_cache lists (transformers <= 4.x),
             (b) modern .layers[i].keys/.values (transformers 5.x DynamicLayer),
             (c) tuple-of-tuples fallback.
    Ref: transformers cache_utils.py DynamicLayer, DynamicCache.
    """
    # (a) Legacy: direct list attributes
    if hasattr(cache, "key_cache") and isinstance(getattr(cache, "key_cache"), list):
        kc = cache.key_cache
        if len(kc) > 0 and hasattr(kc[0], "shape"):
            return kc, cache.value_cache
    # (b) Modern: DynamicLayer objects
    if hasattr(cache, "layers") and len(cache.layers) > 0:
        layer0 = cache.layers[0]
        if hasattr(layer0, "keys") and layer0.keys is not None and layer0.keys.numel() > 0:
            keys = [l.keys for l in cache.layers if hasattr(l, "keys") and l.keys is not None]
            vals = [l.values for l in cache.layers if hasattr(l, "values") and l.values is not None]
            if keys:
                return keys, vals
    # (c) Tuple-of-tuples
    try:
        return [layer[0] for layer in cache], [layer[1] for layer in cache]
    except (TypeError, IndexError):
        raise ValueError(f"Cannot extract KV tensors from cache type: {type(cache)}")

def get_cache_seq_len(cache):
    """Return physical sequence length of KV cache (version-agnostic).
    Ref: transformers Cache.get_seq_length().
    """
    if cache is None:
        return 0
    if hasattr(cache, "get_seq_length"):
        try:
            v = cache.get_seq_length()
            return int(v) if not isinstance(v, int) else v
        except Exception:
            pass
    try:
        keys, _ = _get_kv_tensors(cache)
        if keys:
            return keys[0].shape[-2]
    except Exception:
        pass
    return 0

def _rebuild_cache_from_tensors(key_list, value_list):
    """Rebuild a DynamicCache from raw key/value tensor lists.
    Uses the DynamicCache.update() API to ensure proper DynamicLayer
    initialization — the ONLY safe way to construct a cache that the
    model will correctly append to during subsequent forward passes.
    Ref: DynamicLayer.update() in transformers cache_utils.py performs
    torch.cat([self.keys, key_states], dim=-2) after lazy_initialization.
    """
    from transformers.cache_utils import DynamicCache
    new_cache = DynamicCache()
    for layer_idx in range(len(key_list)):
        new_cache.update(key_list[layer_idx], value_list[layer_idx], layer_idx)
    return new_cache

# -----------------------------------------------------------------------------
# Pager classes (VirtualPageManager, Adaptive, Predictive, RAGAware)
# -----------------------------------------------------------------------------
class VirtualPageManager:
    def __init__(self, threshold_gb: float = 7.5, page_size: int = 1024, sink_size: int = 4):
        self.threshold_bytes  = threshold_gb * (1024 ** 3)
        self.page_size        = page_size
        self.sink_size        = sink_size
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False

    def check_vram_pressure(self) -> bool:
        if not torch.cuda.is_available():
            return False
        return torch.cuda.memory_allocated(0) > self.threshold_bytes

    def offload_kv_cache(self, past_key_values):
        """Offload middle KV tokens to CPU when sequence exceeds page_size.
        Returns cache UNCHANGED when no paging needed — avoids reconstructing
        the DynamicCache, which would break DynamicLayer state tracking.
        When paging IS needed, rebuilds via _rebuild_cache_from_tensors().
        """
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.check_vram_pressure():
            return past_key_values

        key_cache, value_cache = _get_kv_tensors(past_key_values)

        # CRITICAL: Only reconstruct when actual paging is needed.
        # Unnecessary reconstruction resets DynamicLayer internal state,
        # preventing the model from appending new KV states on subsequent
        # forward passes (cf. DynamicLayer.update() in transformers).
        if not any(k.shape[-2] > self.page_size for k in key_cache):
            return past_key_values

        new_keys, new_values = [], []
        layers_paged = 0
        for layer_idx, (k, v) in enumerate(zip(key_cache, value_cache)):
            seq_len = k.shape[-2]
            if seq_len > self.page_size:
                keep_size = self.page_size - self.sink_size
                k_cpu = k[:, :, self.sink_size:-keep_size, :].detach().to('cpu', non_blocking=True)
                v_cpu = v[:, :, self.sink_size:-keep_size, :].detach().to('cpu', non_blocking=True)
                self.cpu_swap_storage.append((layer_idx, k_cpu, v_cpu))
                k_new = torch.cat([k[:, :, :self.sink_size, :], k[:, :, -keep_size:, :]], dim=2)
                v_new = torch.cat([v[:, :, :self.sink_size, :], v[:, :, -keep_size:, :]], dim=2)
                new_keys.append(k_new)
                new_values.append(v_new)
                layers_paged += 1
            else:
                new_keys.append(k)
                new_values.append(v)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        gc.collect(); torch.cuda.empty_cache()

        new_cache = _rebuild_cache_from_tensors(new_keys, new_values)
        if not self._debug_printed:
            phys_len = get_cache_seq_len(new_cache)
            print(f"[Cognito Engine] VirtualPageManager paging activated.")
            print(f"[Cognito Engine] Offloaded {layers_paged} layers | Physical seq len: {phys_len}")
            self._debug_printed = True
        return new_cache

    def reset(self):
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False
        force_cleanup()

    @property
    def blocks_in_cpu(self) -> int:
        return len(self.cpu_swap_storage)

class AdaptiveVirtualPageManager(VirtualPageManager):
    def __init__(self, initial_threshold_gb: float = 7.5, page_size: int = 1024, safety_margin: float = 0.15, ema_alpha: float = 0.3, warmup_steps: int = 10, gpu_capacity_gb: float = 15.5):
        super().__init__(threshold_gb=initial_threshold_gb, page_size=page_size)
        self.safety_margin   = safety_margin
        self.ema_alpha       = ema_alpha
        self.warmup_steps    = warmup_steps
        self.gpu_capacity_gb = gpu_capacity_gb
        self._ema_gb         = None
        self._step           = 0

    def _update_threshold(self):
        if not torch.cuda.is_available():
            return
        current_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
        self._step += 1
        if self._ema_gb is None:
            self._ema_gb = current_gb
        else:
            self._ema_gb = self.ema_alpha * current_gb + (1 - self.ema_alpha) * self._ema_gb
        if self._step > self.warmup_steps:
            adaptive_gb = min(self._ema_gb * (1 + self.safety_margin), self.gpu_capacity_gb * 0.92)
            self.threshold_bytes = adaptive_gb * (1024 ** 3)

    def offload_kv_cache(self, past_key_values):
        self._update_threshold()
        return super().offload_kv_cache(past_key_values)

    def reset(self):
        super().reset()
        self._ema_gb = None
        self._step   = 0

class PredictiveMemoryPolicy(AdaptiveVirtualPageManager):
    NUM_KV_HEADS   = 8
    HEAD_DIM       = 128
    NUM_LAYERS     = 32
    BYTES_PER_ELEM = 2
    KV_FACTOR      = 2

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def predict_kv_increment_gb(self, delta_tokens: int) -> float:
        bytes_cost = (delta_tokens * self.NUM_KV_HEADS * self.HEAD_DIM * self.NUM_LAYERS * self.KV_FACTOR * self.BYTES_PER_ELEM)
        return bytes_cost / (1024 ** 3)

    def should_offload_predictive(self, delta_tokens: int = 1) -> bool:
        if not torch.cuda.is_available(): return False
        current_gb  = torch.cuda.memory_allocated(0) / (1024 ** 3)
        predict_gb  = self.predict_kv_increment_gb(delta_tokens)
        threshold_gb = self.threshold_bytes / (1024 ** 3)
        return (current_gb + predict_gb) > threshold_gb

    def auto_calibrate(self, model_baseline_gb: float, headroom_gb: float = 0.9):
        threshold_gb = model_baseline_gb + headroom_gb
        self.threshold_bytes = threshold_gb * (1024 ** 3)
        print(f"[PredictivePolicy] Threshold auto-calibrado: {threshold_gb:.2f} GB")

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None: return past_key_values
        if not self.should_offload_predictive(delta_tokens=1): return past_key_values
        self.active = True
        return VirtualPageManager.offload_kv_cache(self, past_key_values)

class RAGAwarePager(PredictiveMemoryPolicy):
    class KVSegment:
        def __init__(self, seg_id: int, score: float, start_pos: int, end_pos: int, label: str = "", turn_id: int = 0):
            self.seg_id    = seg_id
            self.score     = score
            self.start_pos = start_pos
            self.end_pos   = end_pos
            self.label     = label
            self.turn_id   = turn_id

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._segments     = []
        self._next_seg_id  = 0
        self._current_pos  = 0
        self._evictions    = 0
        self.global_turn   = 0

    def auto_calibrate(self, model_baseline_gb: float, headroom_gb: float = 0.9):
        threshold_gb = model_baseline_gb + headroom_gb
        self.threshold_bytes = threshold_gb * (1024 ** 3)
        print(f"[RAGAwarePager] Auto-threshold: {model_baseline_gb:.2f} GB + {headroom_gb:.2f} GB = {threshold_gb:.2f} GB")

    def register_segment(self, token_count: int, score: float, label: str = "", turn_id: int = -1) -> int:
        seg = RAGAwarePager.KVSegment(
            seg_id    = self._next_seg_id,
            score     = score,
            start_pos = self._current_pos,
            end_pos   = self._current_pos + token_count,
            label     = label,
            turn_id   = self.global_turn if turn_id == -1 else turn_id,
        )
        self._segments.append(seg)
        self._current_pos += token_count
        self._next_seg_id += 1
        return seg.seg_id

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None: return past_key_values
        if not self.should_offload_predictive(delta_tokens=1): return past_key_values
        if not self._segments: return VirtualPageManager.offload_kv_cache(self, past_key_values)

        def ebbinghaus_score(s):
            lag = self.global_turn - s.turn_id
            return s.score * math.exp(-0.4 * lag)
        self._segments.sort(key=ebbinghaus_score)
        victim = self._segments.pop(0)
        return self._evict_segment(past_key_values, victim)

    def _evict_segment(self, past_key_values, victim):
        """Surgically remove a KV segment and rebuild cache via .update() API.
        Analogous to H2O greedy eviction (Zhang et al., NeurIPS 2023) but
        guided by cross-encoder relevance scores with Ebbinghaus temporal
        decay (cf. TRIM-KV, CAKE/ICLR 2025).
        """
        start, end = victim.start_pos, victim.end_pos
        key_cache, value_cache = _get_kv_tensors(past_key_values)

        new_keys, new_values = [], []
        for layer_idx in range(len(key_cache)):
            k = key_cache[layer_idx]
            v = value_cache[layer_idx]
            seq = k.shape[-2]
            s = min(start, seq)
            e = min(end, seq)
            if s == e:
                new_keys.append(k)
                new_values.append(v)
                continue
            if s == 0:
                k_kept = k[:, :, e:, :]
                v_kept = v[:, :, e:, :]
            elif e >= seq:
                k_kept = k[:, :, :s, :]
                v_kept = v[:, :, :s, :]
            else:
                k_kept = torch.cat([k[:, :, :s, :], k[:, :, e:, :]], dim=2)
                v_kept = torch.cat([v[:, :, :s, :], v[:, :, e:, :]], dim=2)
            new_keys.append(k_kept.contiguous())
            new_values.append(v_kept.contiguous())

        evicted_len = min(end, key_cache[0].shape[-2]) - min(start, key_cache[0].shape[-2])
        # Adjust segment positions (cf. vLLM block table compaction)
        remaining = []
        for seg in self._segments:
            if seg.start_pos >= end:
                seg.start_pos -= evicted_len
                seg.end_pos   -= evicted_len
            elif seg.end_pos > start:
                seg.end_pos = max(seg.start_pos, seg.end_pos - evicted_len)
            if seg.end_pos > seg.start_pos:
                remaining.append(seg)
        self._segments = remaining
        self._current_pos -= evicted_len
        self._evictions   += 1

        # Rebuild using .update() API for DynamicLayer compatibility
        new_cache = _rebuild_cache_from_tensors(new_keys, new_values)
        phys_len = get_cache_seq_len(new_cache)

        if not self._debug_printed:
            print(f"[RAGAwarePager] Eviction #{self._evictions}: segment '{victim.label}' (score={victim.score:.4f}, {evicted_len} tokens)")
            self._debug_printed = True

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        return new_cache

    def reset(self):
        super().reset()
        self._segments.clear()
        self._next_seg_id = 0
        self._current_pos = 0
        self._evictions   = 0

    @property
    def eviction_count(self) -> int: return self._evictions

# -----------------------------------------------------------------------------
# HardenedLLMEngine
# -----------------------------------------------------------------------------
class HardenedLLMEngine:
    def __init__(self, model_name: str, vector_db_path: str, paging_manager=None, top_k_retrieval: int = 10, top_k_rerank: int = 3):
        self.model_name      = model_name
        self.vector_db_path  = vector_db_path
        self.pager           = paging_manager
        self.top_k_retrieval = top_k_retrieval
        self.top_k_rerank    = top_k_rerank
        self.model           = None
        self.tokenizer       = None
        self.collection      = None
        self._embed_model    = None
        self._rerank_model   = None
        self.cached_prefix_kv = None
        self.cached_prefix_len = 0
        self.system_prefix = (
            "<s>[INST] You are a precise factual assistant. "
            "Answer the question using ONLY the provided context. "
            "Give a SHORT, DIRECT answer in 1 to 5 words. "
            "Do not explain or elaborate.\n\n"
        )

    def initialize_runtime(self):
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.35, device=0)
            print("[Emulador TCC] Alocacao PyTorch restrita a 35% de uma T4 (~5.4GB totais).")
        client          = chromadb.PersistentClient(path=self.vector_db_path)
        self.collection = client.get_collection("cognito_knowledge_base")

        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None: self.tokenizer.pad_token = self.tokenizer.eos_token

        try:
            self.model = AutoModelForCausalLM.from_pretrained(self.model_name, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16, attn_implementation="sdpa")
            print("[Cognito Engine] attn_implementation=sdpa ativo.")
        except Exception:
            print("[Cognito Engine] SDPA indisponivel, usando implementacao padrao.")
            self.model = AutoModelForCausalLM.from_pretrained(self.model_name, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16)
        self.model.eval()

        if self.pager is not None and hasattr(self.pager, 'auto_calibrate'):
            baseline_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
            self.pager.auto_calibrate(baseline_gb, headroom_gb=0.9)

        self._embed_model  = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
        self._rerank_model = CrossEncoder(RERANKER_MODEL_NAME)

        print("[Cognito Engine] Compilando Prefix Cache RAG...")
        device = next(self.model.parameters()).device
        prefix_enc = self.tokenizer(self.system_prefix, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model(input_ids=prefix_enc["input_ids"], attention_mask=prefix_enc["attention_mask"], use_cache=True)
            self.cached_prefix_kv = outputs.past_key_values
            self.cached_prefix_len = prefix_enc["input_ids"].shape[-1]
            print(f"[Cognito Engine] Prefix Cache gerado: {self.cached_prefix_len} tokens retidos em MvRAM quente.")

    def retrieve_with_scores(self, query: str) -> list:
        query_vec = self._embed_model.encode([query], normalize_embeddings=True)
        results = self.collection.query(query_embeddings=query_vec.tolist(), n_results=min(self.top_k_retrieval, self.collection.count()))
        candidates = results["documents"][0]
        if not candidates: return []
        pairs  = [(query, doc) for doc in candidates]
        scores = self._rerank_model.predict(pairs)
        ranked = sorted(zip(scores, candidates), reverse=True)
        return [(doc, float(sc)) for sc, doc in ranked[: self.top_k_rerank]]

    def _passage_level_prefill(self, docs_scores: list, question: str, initial_past_kv=None) -> tuple:
        device  = next(self.model.parameters()).device
        past_kv = initial_past_kv
        current_len = self.cached_prefix_len if past_kv is not None else 0

        if past_kv is None:
            prefix_str = self.system_prefix + "Context:\n"
            enc_prefix = self.tokenizer(prefix_str, return_tensors="pt", add_special_tokens=True)
            prefix_ids = enc_prefix["input_ids"].to(device)
            att_mask   = torch.ones((1, prefix_ids.shape[-1]), dtype=torch.long, device=device)
            with torch.no_grad():
                out = self.model(input_ids=prefix_ids, attention_mask=att_mask, use_cache=True)
            past_kv = out.past_key_values
            current_len = prefix_ids.shape[-1]
        else:
            ctx_str = "Context:\n"
            enc_ctx = self.tokenizer(ctx_str, return_tensors="pt", add_special_tokens=False)
            ctx_ids = enc_ctx["input_ids"].to(device)
            att_mask = torch.ones((1, current_len + ctx_ids.shape[-1]), dtype=torch.long, device=device)
            with torch.no_grad():
                out = self.model(input_ids=ctx_ids, attention_mask=att_mask, past_key_values=past_kv, use_cache=True)
            past_kv = out.past_key_values
            current_len += ctx_ids.shape[-1]

        if isinstance(self.pager, RAGAwarePager):
            self.pager.reset()
            self.pager.active = True

        # Per-query min-max normalization of cross-encoder scores.
        # Maps best passage -> 1.0, worst -> 0.0. Distribution-agnostic,
        # unlike a fixed score shift which compresses the informative tail.
        if docs_scores:
            raw = [sc for _, sc in docs_scores]
            _min, _max = min(raw), max(raw)
            if _max > _min:
                docs_scores = [(d, (s - _min) / (_max - _min)) for d, s in docs_scores]
            else:
                docs_scores = [(d, 1.0) for d, s in docs_scores]

        for i, (passage, score) in enumerate(docs_scores):
            passage_text = passage + "\n" if i < len(docs_scores) - 1 else passage
            enc = self.tokenizer(passage_text, return_tensors="pt", add_special_tokens=False)
            passage_ids = enc["input_ids"].to(device)
            n_tokens = passage_ids.shape[-1]
            att_mask = torch.ones((1, current_len + n_tokens), dtype=torch.long, device=device)
            with torch.no_grad():
                out = self.model(input_ids=passage_ids, attention_mask=att_mask, past_key_values=past_kv, use_cache=True)
            past_kv = out.past_key_values
            current_len += n_tokens
            if isinstance(self.pager, RAGAwarePager):
                self.pager.register_segment(token_count=n_tokens, score=score, label=f"passage_{i}_score{score:.3f}")
                # NOTE: Eviction deferred to decode phase.
                # Calling offload here triggers VPM fallback (no segments exceed page_size)
                # which reconstructs the DynamicCache, resetting internal state and
                # preventing KV accumulation. See vLLM chunked-prefill (Agrawal et al., 2024).
        print(f"[Cognito Engine] Prefill complete: {get_cache_seq_len(past_kv)} tokens cached for {len(docs_scores)} passages")

        q_str = f"\nQuestion: {question} [/INST]"
        enc_q = self.tokenizer(q_str, return_tensors="pt", add_special_tokens=False)
        q_ids = enc_q["input_ids"].to(device)
        n_q = q_ids.shape[-1]
        att_mask = torch.ones((1, current_len + n_q), dtype=torch.long, device=device)
        with torch.no_grad():
            out = self.model(input_ids=q_ids, attention_mask=att_mask, past_key_values=past_kv, use_cache=True)
        past_kv = out.past_key_values
        last_token_id = q_ids[0, -1]
        return past_kv, att_mask, last_token_id

    def _generate_token_by_token(self, input_ids, attention_mask, max_new_tokens, initial_past_kv=None):
        device = input_ids.device
        past_kv = initial_past_kv
        generated_ids = input_ids.clone()
        current_mask  = attention_mask.clone()
        if self.pager:
            if initial_past_kv is None: self.pager.reset()
            self.pager.active = True
        ttft = 0.0
        decode_times = []
        t0_gen = time.time()
        with torch.no_grad():
            for step in range(max_new_tokens):
                t_iter_start = time.time()
                model_input = generated_ids[:, -1:] if (step > 0 or past_kv is not None) else generated_ids
                outputs = self.model(input_ids=model_input, attention_mask=current_mask, past_key_values=past_kv, use_cache=True)
                next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                past_kv    = outputs.past_key_values
                if self.pager:
                    past_kv = self.pager.offload_kv_cache(past_kv)
                    # Sincroniza a máscara de atenção caso o pager tenha evictado tokens
                    new_seq_len = get_cache_seq_len(past_kv)
                    if new_seq_len < current_mask.shape[-1]:
                        current_mask = current_mask[:, -new_seq_len:]
                
                generated_ids = torch.cat([generated_ids, next_token], dim=-1)
                current_mask  = torch.cat([current_mask, torch.ones((1, 1), dtype=torch.long, device=device)], dim=-1)
                t_iter_end = time.time()
                if step == 0: ttft = t_iter_end - t0_gen
                else: decode_times.append(t_iter_end - t_iter_start)
                if next_token.item() == self.tokenizer.eos_token_id: break
        itl_avg = sum(decode_times) / len(decode_times) if decode_times else 0.0
        return generated_ids, ttft, itl_avg

    def _generate_hybrid(self, input_ids, attention_mask, max_new_tokens, use_paging, use_chunked_prefill=False, initial_past_kv=None):
        seq_len = input_ids.shape[-1]
        if not use_paging or self.pager is None or seq_len < PAGING_CONTEXT_THRESHOLD_TOKENS:
            if self.pager: self.pager.reset(); self.pager.active = False
            t0_fast = time.time()
            with torch.no_grad():
                generated_ids = self.model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, use_cache=True, pad_token_id=self.tokenizer.pad_token_id, past_key_values=initial_past_kv)
            latency = time.time() - t0_fast
            n_tokens = generated_ids.shape[-1] - input_ids.shape[-1]
            return generated_ids, "fast_path", latency * 0.8, (latency * 0.2) / n_tokens if n_tokens > 0 else 0
        generated_ids, ttft, itl_avg = self._generate_token_by_token(input_ids, attention_mask, max_new_tokens, initial_past_kv=initial_past_kv)
        return generated_ids, "paging_path", ttft, itl_avg

    def generate(self, query: str, retrieved_context: str = "", use_rag: bool = True, max_new_tokens: int = MAX_NEW_TOKENS, use_paging: bool = True, use_chunked_prefill: bool = False, use_prefix_caching: bool = False, use_passage_prefill: bool = False, docs_scores: list = None) -> dict:
        device = next(self.model.parameters()).device
        if use_passage_prefill and use_rag and docs_scores:
            initial_past_kv = self.cached_prefix_kv if use_prefix_caching else None
            past_kv, att_mask, last_token_id = self._passage_level_prefill(docs_scores, query, initial_past_kv)
            input_ids = last_token_id.unsqueeze(0).unsqueeze(0)
            torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            try:
                generated_ids, ttft, itl = self._generate_token_by_token(input_ids=input_ids, attention_mask=att_mask, max_new_tokens=max_new_tokens, initial_past_kv=past_kv)
                status = "SUCCESS"
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    force_cleanup()
                    return {"response": "", "latency_sec": 0, "throughput_tps": 0, "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024 ** 3, "blocks_paged": 0, "gen_path": "oom", "status": "OOM_FAILURE", "ttft_sec": 0.0, "itl_sec": 0.0}
                raise
            latency = time.time() - t0
            n_new = generated_ids.shape[-1] - 1
            response = self.tokenizer.decode(generated_ids[0, 1:], skip_special_tokens=True)
            return {"response": response, "latency_sec": latency, "throughput_tps": n_new / latency if latency > 0 else 0.0, "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024 ** 3, "blocks_paged": self.pager.blocks_in_cpu if self.pager else 0, "gen_path": "passage_prefill_path", "ttft_sec": ttft, "itl_sec": itl, "status": status}

        context_text = (retrieved_context[:MAX_CONTEXT_CHARS] + "... [TRUNCATED]") if len(retrieved_context) > MAX_CONTEXT_CHARS else retrieved_context
        if use_prefix_caching and self.cached_prefix_kv is not None:
            dynamic_prompt = f"Context:\n{context_text}\n\nQuestion: {query} [/INST]"
            enc_dyn = self.tokenizer(dynamic_prompt, return_tensors="pt", add_special_tokens=False)
            dynamic_ids = enc_dyn["input_ids"].to(device)
            kc, vc = _get_kv_tensors(self.cached_prefix_kv)
            new_cache = _rebuild_cache_from_tensors([k.clone() for k in kc], [v.clone() for v in vc])
            input_ids, initial_past_kv = dynamic_ids, new_cache
            attention_mask = torch.ones((1, self.cached_prefix_len + dynamic_ids.shape[-1]), dtype=torch.long, device=device)
        else:
            prompt = self.system_prefix + f"Context:\n{context_text}\n\nQuestion: {query} [/INST]"
            enc = self.tokenizer(prompt, return_tensors="pt"); input_ids, attention_mask = enc["input_ids"].to(device), enc["attention_mask"].to(device); initial_past_kv = None
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()
        try:
            generated_ids, gen_path, ttft, itl = self._generate_hybrid(input_ids, attention_mask, max_new_tokens, use_paging, use_chunked_prefill, initial_past_kv=initial_past_kv); status = "SUCCESS"
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                force_cleanup()
                return {"response": "", "latency_sec": 0, "throughput_tps": 0, "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024 ** 3, "blocks_paged": 0, "gen_path": "oom", "status": "OOM_FAILURE", "ttft_sec": 0.0, "itl_sec": 0.0}
            raise
        latency = time.time() - t0; n_new = generated_ids.shape[-1] - input_ids.shape[-1]
        response = self.tokenizer.decode(generated_ids[0, input_ids.shape[-1]:], skip_special_tokens=True)
        return {"response": response, "latency_sec": latency, "throughput_tps": n_new / latency if latency > 0 else 0.0, "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024 ** 3, "blocks_paged": self.pager.blocks_in_cpu if self.pager else 0, "gen_path": gen_path, "ttft_sec": ttft, "itl_sec": itl, "status": status}

# -----------------------------------------------------------------------------
# Métricas e Avaliação
# -----------------------------------------------------------------------------
import re, string; from collections import Counter
def normalize_answer(s):
    return ' '.join(re.sub(r'\b(a|an|the)\b', ' ', s.lower()).split()).translate(str.maketrans('', '', string.punctuation))
def exact_match_score(prediction, ground_truth): return normalize_answer(ground_truth) in normalize_answer(prediction)
def f1_score(prediction, ground_truth):
    p, t = normalize_answer(prediction).split(), normalize_answer(ground_truth).split()
    if not p or not t: return 0.0
    common = Counter(p) & Counter(t); n = sum(common.values())
    if n == 0: return 0.0
    prec, rec = n / len(p), n / len(t); return (2 * prec * rec) / (prec + rec)
def metric_max_over_ground_truths(m_fn, pred, truths): return max(m_fn(pred, gt) for gt in truths) if truths else 0.0
def run_evaluation(queries, configs, engine, gold_answers, max_new_tokens=MAX_NEW_TOKENS, pre_fetched=None, pre_fetched_scores=None):
    if pre_fetched is None:
        print("\n[Ablacao] Pre-computando Retrieval..."); pre_fetched, pre_fetched_scores = {}, {}
        for q in queries:
            ds = engine.retrieve_with_scores(q); pre_fetched_scores[q] = ds; pre_fetched[q] = "\n".join([d for d, _ in ds])
    if engine._embed_model: del engine._embed_model; engine._embed_model = None
    if engine._rerank_model: del engine._rerank_model; engine._rerank_model = None
    force_cleanup(); all_results = {}
    for cfg in configs:
        label = cfg["label"]; print(f"\n{'='*60}\nConfiguracao: {label}\n{'='*60}")
        if cfg.get("use_paging") and cfg.get("threshold_gb"): engine.pager.threshold_bytes = cfg["threshold_gb"] * (1024 ** 3)
        records = []
        for i, q in enumerate(queries):
            print(f"  [{i+1}/{len(queries)}] {q[:70]}")
            m = engine.generate(query=q, retrieved_context=pre_fetched.get(q, ""), use_rag=cfg.get("use_rag", True), max_new_tokens=max_new_tokens, use_paging=cfg.get("use_paging", False), use_chunked_prefill=cfg.get("use_chunked_prefill", False), use_prefix_caching=cfg.get("use_prefix_caching", False), use_passage_prefill=cfg.get("use_passage_prefill", False), docs_scores=pre_fetched_scores.get(q, []))
            m["em_score"] = metric_max_over_ground_truths(exact_match_score, m["response"], gold_answers.get(q, [])) if m["status"] == "SUCCESS" else None
            m["f1_tok"] = metric_max_over_ground_truths(f1_score, m["response"], gold_answers.get(q, [])) if m["status"] == "SUCCESS" else None
            m["query"] = q; records.append(m)
            print(f"     Status: {m['status']} | VRAM: {m['peak_vram_gb']:.2f}GB | TTFT: {m.get('ttft_sec',0):.2f}s | EM: {m['em_score']} | Paged: {m['blocks_paged']}")
            force_cleanup()
        all_results[label] = records
    print(f"\n{'Configuracao':<45} {'VRAM Max'} {'TTFT(s)'} {'ITL(ms)'} {'Acc (EM)'} {'OOMs'}")
    for label, recs in all_results.items():
        ok = [r for r in recs if r["status"] == "SUCCESS"]; ooms = len(recs) - len(ok)
        if not ok: print(f"  {label:<40} OOM total"); continue
        max_vram = max(r["peak_vram_gb"] for r in ok); avg_ttft = sum(r["ttft_sec"] for r in ok) / len(ok); avg_itl = sum(r["itl_sec"] for r in ok) * 1000 / len(ok); avg_em = sum(r["em_score"] for r in ok) * 100 / len(ok)
        print(f"  {label:<45} {max_vram:>8.2f} {avg_ttft:>7.2f}s {avg_itl:>7.0f}ms {avg_em:>8.1f}% {ooms:>4}")
    return all_results

def create_long_context(engine, base_passage: str, target_tokens: int = 8192) -> str:
    enc = engine.tokenizer(base_passage, return_tensors="pt", add_special_tokens=False)
    single_len = enc["input_ids"].shape[-1]
    if single_len == 0: return base_passage * 1000
    repeats = (target_tokens // single_len) + 1
    return ((base_passage + " ") * repeats)[: target_tokens * 4]

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    import sys
    is_test = "--test" in sys.argv
    if not os.path.exists("gold_answers.json"):
        print("AVISO: gold_answers.json não encontrado. Certifique-se de executar a ingestão (Cell 5) primeiro.")
        return
    with open("gold_answers.json", "r") as f: gold_answers = json.load(f)
    eval_queries = list(gold_answers.keys())[:(2 if is_test else NUM_QUERIES_EVAL)]
    pager = RAGAwarePager(initial_threshold_gb=7.5, page_size=1024, safety_margin=0.15, ema_alpha=0.3, warmup_steps=10, gpu_capacity_gb=15.5)
    engine = HardenedLLMEngine(model_name=LLM_MODEL_NAME, vector_db_path=VECTOR_DB_PATH, paging_manager=pager)
    engine.initialize_runtime()

    CONFIGURATIONS = [
        {"label": "C1: No RAG, No Paging", "use_rag": False, "use_paging": False},
        {"label": "C6: RAG + RAGAware Pager", "use_rag": True, "use_paging": True, "use_passage_prefill": True}
    ]
    
    # IMPORTANTE: Extrair contexto do stress test ANTES do primeiro run_evaluation, 
    # pois a função run_evaluation deleta os modelos de embedding/rerank para liberar VRAM!
    stress_queries = eval_queries[:10]
    ds_stress = engine.retrieve_with_scores(stress_queries[0]) if len(stress_queries) > 0 else []
    long_ctx = create_long_context(engine, ds_stress[0][0] if ds_stress else "Test context", 8000)
    pre_f_stress = {q: long_ctx for q in stress_queries}
    pre_s_stress = {q: [(long_ctx, 0.5)] for q in stress_queries}

    run_evaluation(eval_queries, CONFIGURATIONS, engine, gold_answers)

    print("\n\n>>> INICIANDO TESTES DE ESTRESSE COM VRAM REDUZIDA (25%) <<<")
    torch.cuda.set_per_process_memory_fraction(0.25, device=0)
    stress_pager = PredictiveMemoryPolicy(initial_threshold_gb=3.5, gpu_capacity_gb=3.8)
    stress_pager.auto_calibrate(3.86, 0.4)
    engine.pager = stress_pager
    
    STRESS_CFGS = [
        {"label": "S1: Stress - No Paging (8k tokens)", "use_paging": False},
        {"label": "S2: Stress - Predictive Paging (8k tokens)", "use_paging": True}
    ]
    run_evaluation(stress_queries, STRESS_CFGS, engine, gold_answers, pre_fetched=pre_f_stress, pre_fetched_scores=pre_s_stress)

if __name__ == "__main__": main()


In [ ]:
!uv run 3_inferencia.py --test
!uv run 3_inferencia.py

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE FIM DE SESSÃO — Execute ANTES de fechar o Colab
# ═══════════════════════════════════════════════════════════════
# Persiste dados locais de volta ao Drive.
# IMPORTANTE: rodar antes de encerrar a sessão ou antes do timeout.

import os, shutil, time, json as _j

DRIVE_ROOT = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT = '/content'

os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

saved = []

# ── Salva ChromaDB ────────────────────────────────────────────────
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
if os.path.exists(f'{LOCAL_CHROMA}/chroma.sqlite3'):
    print('Salvando ChromaDB → Drive ...')
    t0 = time.time()
    if os.path.exists(DRIVE_CHROMA):
        shutil.rmtree(DRIVE_CHROMA)
    shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DRIVE_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB salvo: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
    saved.append('chroma_cognito')
else:
    print('  ⚠ ChromaDB não encontrado em /content/ — nada a salvar.')

# ── Salva gold_answers.json ───────────────────────────────────────
LOCAL_GOLD = f'{LOCAL_ROOT}/gold_answers.json'
if os.path.exists(LOCAL_GOLD):
    shutil.copy(LOCAL_GOLD, f'{DRIVE_ROOT}/gold_answers.json')
    print(f'  ✓ gold_answers.json salvo.')
    saved.append('gold_answers.json')

# ── Salva checkpoints JSONL ───────────────────────────────────────
ckpt_files = [f for f in os.listdir(LOCAL_ROOT)
              if f.startswith('results_checkpoint_') and f.endswith('.jsonl')]
for ck in ckpt_files:
    src = f'{LOCAL_ROOT}/{ck}'
    dst = f'{DRIVE_ROOT}/checkpoints/{ck}'
    shutil.copy(src, dst)
    # Conta linhas (queries completadas)
    n_lines = sum(1 for _ in open(src))
    print(f'  ✓ {ck}: {n_lines} queries salvas.')
    saved.append(ck)

# ── Relatório ─────────────────────────────────────────────────────
print()
if saved:
    print(f'✓ Sessão salva no Drive: {saved}')
    print(f'  Caminho: {DRIVE_ROOT}')
else:
    print('⚠ Nada foi salvo — verifique os caminhos.')

# ── Status dos checkpoints (progresso da ablação) ─────────────────
print()
print('Status da Ablação:')
all_ckpts = [f for f in os.listdir(f'{DRIVE_ROOT}/checkpoints') if f.endswith('.jsonl')]
if all_ckpts:
    for ck in sorted(all_ckpts):
        path = f'{DRIVE_ROOT}/checkpoints/{ck}'
        lines = open(path).readlines()
        ooms = sum(1 for l in lines if '"OOM_FAILURE"' in l)
        ok = len(lines) - ooms
        print(f'  {ck:<45} {ok:>3} OK  {ooms:>3} OOM  ({len(lines):>3} total)')
else:
    print('  Nenhum checkpoint encontrado ainda.')